# Interactive FSM-PK Visualization Using Google Earth Engine

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/waleedgeo/FSM-PK/blob/main/notebooks/02_gee_interactive_map.ipynb)

This notebook demonstrates interactive visualization and spatial analysis of the Pakistan Flood Susceptibility Map (FSM-PK) using Google Earth Engine.

## Objectives

1. Initialize and authenticate with Google Earth Engine
2. Load FSM-PK assets (LGBM and XGBoost models)
3. Create split-panel interactive maps
4. Compare LGBM vs XGBoost model outputs
5. Calculate spatial statistics

## About the Dataset

- **Resolution**: 30 meters
- **Models**: LGBM (0.85 accuracy) and XGBoost (0.82 accuracy)
- **Classes**: 1-5 (Very Low to Very High susceptibility)
- **GEE Assets**:
  - LGBM: `projects/waleedgeo/assets/fsm_pk_lgbm`
  - XGBoost: `projects/waleedgeo/assets/fsm_pk_xgboost`

## Citation

```
Waleed, M., & Sajjad, M. (2025). High-resolution flood susceptibility mapping
and exposure assessment in Pakistan: An integrated artificial intelligence,
machine learning and geospatial framework. International Journal of Disaster
Risk Reduction, 121, 105442. https://doi.org/10.1016/j.ijdrr.2025.105442
```

---

## 1. Setup and Installation

Install required packages (only needed in Google Colab):

In [ ]:
# Uncomment and run if using Google Colab
# !pip install geemap earthengine-api -q

### Import Libraries

In [1]:
import ee
import geemap

print('All packages imported successfully!')

All packages imported successfully!


---

## 2. Google Earth Engine Authentication

### Authenticate (First Time Only)

Run this cell ONLY once to authenticate with your Google account:

In [ ]:
# Uncomment and run ONLY if this is your first time using GEE
# ee.Authenticate()

### Initialize Earth Engine

Run this cell every time you start the notebook:

In [4]:
try:
    ee.Initialize(project='waleedgeo')
    print('Earth Engine initialized successfully!')
except Exception as e:
    print(f'Initialization failed: {e}')
    print('\nPlease run ee.Authenticate() first')

Earth Engine initialized successfully!


---

## 3. Load FSM-PK Assets

In [19]:
fsm_lgbm = ee.Image('projects/waleedgeo/assets/fsm_pk_lgbm').setDefaultProjection('EPSG:4326', None, 30)
fsm_xgboost = ee.Image('projects/waleedgeo/assets/fsm_pk_xgboost').setDefaultProjection('EPSG:4326', None, 30)


# Flood susceptibility classes
FSM_CLASSES = {1: 'Very Low', 2: 'Low', 3: 'Moderate', 4: 'High', 5: 'Very High'}

# Color palette (green to red)
FSM_COLORS = ['2E7D32', '7CB342', 'FDD835', 'FB8C00', 'C62828']

# Visualization parameters
vis_params = {'min': 1, 'max': 5, 'palette': FSM_COLORS}

Map = geemap.Map()
Map.add_basemap('HYBRID')


# add Pakistan outline
pak_adm0 = ee.FeatureCollection('projects/pak-var/assets/pak_adm0')

Map.addLayer(pak_adm0, {}, 'PAK ADM0')

Map.addLayer(fsm_lgbm, {'min':1, 'max':5, 'palette':FSM_COLORS}, 'FSM PK (LGBM)')
Map.addLayer(fsm_xgboost, {'min':1, 'max':5, 'palette':FSM_COLORS}, 'FSM PK (XGB)', False)

Map.centerObject(fsm_lgbm)

# Add legend
legend_dict = {
    '1 - Very Low': '2E7D32',
    '2 - Low': '7CB342',
    '3 - Moderate': 'FDD835',
    '4 - High': 'FB8C00',
    '5 - Very High': 'C62828'
}
Map.add_legend(title='FSM', legend_dict=legend_dict)


Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

### Map 3: Model Comparison (LGBM vs XGBoost)

In [23]:
# Create split map for model comparison
Map3 = geemap.Map(center=[30.3753, 69.3451], zoom=5)
Map3.add_basemap('HYBRID')

# Create layers
# add Pakistan outline
pak_adm0 = ee.FeatureCollection('projects/pak-var/assets/pak_adm0')

Map3.addLayer(pak_adm0, {}, 'PAK ADM0')

lgbm_layer = geemap.ee_tile_layer(fsm_lgbm, vis_params, 'LGBM (Acc: 0.85)')
xgb_layer = geemap.ee_tile_layer(fsm_xgboost, vis_params, 'XGBoost (Acc: 0.82)')

# Create split panel
Map3.split_map(lgbm_layer, xgb_layer)
Map3.add_legend(title='Flood Susceptibility', legend_dict=legend_dict)

# Display map
Map3

Map(center=[30.3753, 69.3451], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zo…

---

## 6. Spatial Analysis

### Calculate Area of Very High Susceptibility

In [24]:
# Define Pakistan boundary (approximate)
pakistan = ee.Geometry.Rectangle([60.87, 23.69, 77.84, 37.13])

# Create mask for Very High susceptibility (class 5)
very_high_mask = fsm_lgbm.eq(5)

# Calculate area
very_high_area = very_high_mask.multiply(ee.Image.pixelArea()).divide(1e6)

print('Calculating area of Very High susceptibility...\n')
print('(This may take 10-30 seconds)\n')

area_stats = very_high_area.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=pakistan,
    scale=30,
    maxPixels=1e10
)

total_very_high_area = area_stats.getInfo()['b1']

print('=' * 60)
print('VERY HIGH SUSCEPTIBILITY STATISTICS (LGBM Model)')
print('=' * 60)
print(f'\nTotal Very High susceptibility area: {total_very_high_area:,.2f} km²')
print('\nThis represents areas at the highest risk of flooding.')
print('=' * 60)

Calculating area of Very High susceptibility...

(This may take 10-30 seconds)

VERY HIGH SUSCEPTIBILITY STATISTICS (LGBM Model)

Total Very High susceptibility area: 116,057.32 km²

This represents areas at the highest risk of flooding.


---

## Additional Resources

- **Paper**: [https://doi.org/10.1016/j.ijdrr.2025.105442](https://doi.org/10.1016/j.ijdrr.2025.105442)
- **Data**: [https://doi.org/10.5281/zenodo.18513601](https://doi.org/10.5281/zenodo.18513601)
- **Interactive App**: [https://waleedgeo-ee.projects.earthengine.app/view/fsm-pk](https://waleedgeo-ee.projects.earthengine.app/view/fsm-pk)
- **GitHub**: [https://github.com/waleedgeo/FSM-PK](https://github.com/waleedgeo/FSM-PK)

---

## Contact

**Mirza Waleed**
- Email: waleedgeo@outlook.com
- Website: [waleedgeo.com](https://waleedgeo.com)
- LinkedIn: [linkedin.com/in/waleedgeo](https://www.linkedin.com/in/waleedgeo)